# Aim of the notebook
In this notebook we will convert a SpatialData object to an anndata containing sufficient information to perform spatial RNA velocity

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import zarr
import spatialdata as sd
import spatialdata_io as sio
import os

We define the path where the SpatialData object is stored (as zarr)

In [3]:
transcripts=pd.read_parquet('/media/sergio/GBX/Data_for_stvelo/raw_data/DevLung/GW_15/GSM8217893_GW_15_transcripts.parquet')

In [4]:
max_dist_to_nucleus=50
tr = transcripts[transcripts["cell_id"] != "UNASSIGNED" ]
tr.reset_index(inplace=True, drop=True)
# keep only counts situated at less than a certain distance to the nuclei edge
if max_dist_to_nucleus!='all':
    tr=tr[tr['nucleus_distance']<max_dist_to_nucleus]
    
# divide data in nuclei and cytopasm
tn = tr[tr['overlaps_nucleus'] == 1]
tc = tr[tr['overlaps_nucleus'] == 0]

In [5]:
nuc = pd.crosstab(tn['cell_id'],tn['feature_name'])
cyt = pd.crosstab(tc['cell_id'],tc['feature_name'])
# filer adata to keep only common cells 
cell_tr = pd.crosstab(tr['cell_id'],tr['feature_name'])

In [6]:
adata=sc.AnnData(cell_tr)

In [7]:
adata.obs.index

Index(['aaaaakoj-1', 'aaaaceno-1', 'aaaacgga-1', 'aaaacnld-1', 'aaaadkim-1',
       'aaaahdjf-1', 'aaaahokj-1', 'aaaakeab-1', 'aaaalpmp-1', 'aaaamoli-1',
       ...
       'oimffcdf-1', 'oimfhokj-1', 'oimfikdm-1', 'oimfjfkk-1', 'oimflhbe-1',
       'oimfnjif-1', 'oimfnkhi-1', 'oimfnpoh-1', 'oimfolca-1', 'oimfpikl-1'],
      dtype='object', name='cell_id', length=863857)

In [8]:
adata.var['gene_name']=adata.var.index
adata.obs.index = adata.obs.index.astype(str)
# eliminate genes  that are not present in nuc and cyt
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata = adata[:,adata.var['gene_name'].isin(nuc.columns)]
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata = adata[:,adata.var['gene_name'].isin(cyt.columns)]
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
nuc.index=nuc.index.astype(str)
cyt.index=cyt.index.astype(str)

adata.obs['cell_id']=adata.obs.index
#get the cells which has a transcripts mappep to it 
adata = adata[adata.obs['cell_id'].isin(nuc.index)]
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata = adata[adata.obs['cell_id'].isin(cyt.index)]

# create "spliced", "unspliced" layers since scvelo looks for them
nucsort = nuc.loc[adata.obs['cell_id'],adata.var["gene_name"]]
cytsort = cyt.loc[adata.obs['cell_id'],adata.var["gene_name"]]

# define nuclear and cytoplasmic counts in layers. Nuclear counts are stores in 'unspliced' and cytoplasmic counts are stored in 'spliced'
adata.layers['spliced'] = np.array(cytsort)
adata.layers['unspliced'] = np.array(nucsort)
# the counts of all cells are considered the sum of spliced and unspliced
adata.X=adata.layers['spliced']+adata.layers['unspliced']
tr2=tr.loc[:,['cell_id','x_location','y_location']]
meantr=tr2.groupby('cell_id').mean()
meantr.index=meantr.index.astype(str)
meantrsorted=meantr.loc[adata.obs.index,:]
adata.obs['x_centroid']=meantrsorted['x_location']
adata.obs['y_centroid']=meantrsorted['y_location']

/tmp/ipykernel_178156/2297847650.py:7: ImplicitModificationWarning: Trying to modify index of attribute `.obs` of view, initializing view as actual.
  adata.obs.index = adata.obs.index.astype(str)
/tmp/ipykernel_178156/2297847650.py:10: ImplicitModificationWarning: Trying to modify index of attribute `.obs` of view, initializing view as actual.
  adata.obs.index = adata.obs.index.astype(str)
/tmp/ipykernel_178156/2297847650.py:18: ImplicitModificationWarning: Trying to modify index of attribute `.obs` of view, initializing view as actual.
  adata.obs.index = adata.obs.index.astype(str)
/tmp/ipykernel_178156/2297847650.py:27: ImplicitModificationWarning: Setting element `.layers['spliced']` of view, initializing view as actual.
  adata.layers['spliced'] = np.array(cytsort)


# Define paths

In [11]:
path_to_write='/media/sergio/GBX/Data_for_stvelo/raw_data/DevLung/GW_15'

In [10]:
del tr
del tr2
del nuc
del cyt
del nucsort
del cytsort

In [12]:
adata.write(os.path.join(path_to_write,'adata.h5ad'))